# Runtime tables notebook

This notebook generates the runtime-scaling tables. 


In [ ]:
import importlib
import pickle
from pathlib import Path

from IPython.display import Markdown

import adaptive_low_rank.plotting as plotting

importlib.reload(plotting)

repo_root = Path.cwd()
if not (repo_root / "results").exists():
    repo_root = repo_root.parent

results_dir = repo_root / "results" / "runtime_scaling_v2"
results_path = results_dir / "runtime_scaling.pkl"

with results_path.open("rb") as file:
    runtime_results = pickle.load(file)


In [ ]:
tables_dir = results_dir / "tables"
tables_dir.mkdir(parents=True, exist_ok=True)

column_sweep = runtime_results["column_sweep"]
row_sweep = runtime_results["row_sweep"]

fixed_d_tables = {}
for d in column_sweep["dimensions"]:
    mean_df, std_df = plotting.runtime_table(column_sweep, fixed_d=d)
    fixed_d_tables[d] = mean_df
    mean_df.to_csv(tables_dir / f"runtime_fixed_d{d}_mean.csv")
    std_df.to_csv(tables_dir / f"runtime_fixed_d{d}_std.csv")

    caption = f"Relative runtime (s), mean \u00b1 std over repeats, d = {d}"
    styled = plotting.style_runtime_table(
        mean_df, std_df, sig_figs=3, caption=caption
    )
    latex = styled.to_latex(hrules=True, convert_css=True).replace(
        "\u00b1", r"$\pm$"
    )
    (tables_dir / f"runtime_fixed_d{d}.tex").write_text(latex)

    display(Markdown(f"### d = {d}"))
    display(styled)

In [ ]:
fixed_n_tables = {}
for n in row_sweep["n_values"]:
    mean_df, std_df = plotting.runtime_table(row_sweep, fixed_n=n)
    fixed_n_tables[n] = mean_df
    mean_df.to_csv(tables_dir / f"runtime_fixed_n{n}_mean.csv")
    std_df.to_csv(tables_dir / f"runtime_fixed_n{n}_std.csv")

    caption = f"Relative runtime (s), mean \u00b1 std over repeats, n = {n:,}"
    styled = plotting.style_runtime_table(
        mean_df, std_df, sig_figs=3, caption=caption
    )
    latex = styled.to_latex(hrules=True, convert_css=True).replace(
        "\u00b1", r"$\pm$"
    )
    (tables_dir / f"runtime_fixed_n{n}.tex").write_text(latex)

    display(Markdown(f"### n = {n}"))
    display(styled)